In [1]:
#Установка PySpark и настройка окружения
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

import findspark
findspark.init()

from pyspark.sql import SparkSession

# Создание Spark сессии
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("BigData_Lab") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print("Spark запущен")

Spark запущен


In [2]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

trips_path = "trip.csv"
stations_path = "station.csv"

# Загрузка данных
trips_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .csv(trips_path)

stations_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(stations_path)

Задача 1: Найти велосипед с максимальным временем пробега

In [4]:
# Группируем по bike_id и суммируем время
bike_total_time = trips_df.groupBy("bike_id") \
    .agg(
        sum("duration").alias("total_seconds"),
        count("id").alias("num_trips")
    ) \
    .orderBy(col("total_seconds").desc())

# Находим велосипед с максимальным временем
max_bike = bike_total_time.first()

print(f"Велосипед с максимальным временем пробега: {max_bike['bike_id']}")
print(f"Общее время пробега: {max_bike['total_seconds']:,} секунд")

Велосипед с максимальным временем пробега: 535
Общее время пробега: 18,611,693 секунд


Задача 2: Найти наибольшее геодезическое расстояние между станциями

In [6]:
import math
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import udf

# Функция для вычисления гаверсинуса (геодезического расстояния)
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Радиус Земли в километрах

    # Переводим градусы в радианы
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])

    # Разности координат
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    # Формула гаверсинуса
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))

    return R * c

# Регистрируем функцию как UDF
haversine_udf = udf(haversine_distance, DoubleType())

# Подготавливаем данные
stations1 = stations_df.select(
    col("id").alias("station1_id"),
    col("name").alias("station1_name"),
    col("lat").alias("lat1"),
    col("long").alias("lon1")
)

stations2 = stations_df.select(
    col("id").alias("station2_id"),
    col("name").alias("station2_name"),
    col("lat").alias("lat2"),
    col("long").alias("lon2")
)

# Создаем декартово произведение станций (все пары)
station_pairs = stations1.crossJoin(stations2) \
    .filter(col("station1_id") < col("station2_id"))

# Рассчитываем расстояние для каждой пары
station_distances = station_pairs \
    .withColumn("distance_km",
                haversine_udf(col("lat1"), col("lon1"), col("lat2"), col("lon2")))

# Находим максимальное расстояние
max_distance = station_distances.orderBy(col("distance_km").desc()).first()

print(f"Станция 1: {max_distance['station1_name']} (ID: {max_distance['station1_id']})")
print(f"Станция 2: {max_distance['station2_name']} (ID: {max_distance['station2_id']})")
print(f"Расстояние: {max_distance['distance_km']:.2f} км")

Станция 1: SJSU - San Salvador at 9th (ID: 16)
Станция 2: Embarcadero at Sansome (ID: 60)
Расстояние: 69.92 км


Задача 3: Найти путь велосипеда с максимальным временем пробега через станции

In [7]:
# Используем bike_id, найденный в задаче 1
max_bike_id = bike_total_time.first()['bike_id']

# Фильтруем поездки по времени
bike_trips = trips_df \
    .filter(col("bike_id") == max_bike_id) \
    .orderBy("start_date") \
    .select(
        "start_date",
        "end_date",
        "start_station_name",
        "end_station_name",
        "duration"
    )

total_trips = bike_trips.count()
print(f"Количество поездок: {total_trips}")

# Выводим маршрут
print("Первые 35 поездок:")
bike_trips.show(35, truncate=50)

# Статистика по маршруту
unique_stations = bike_trips.select("start_station_name", "end_station_name") \
    .distinct() \
    .count()

print(f"Уникальных станций посещено: {unique_stations}")

Количество поездок: 1328
Первые 35 поездок:
+---------------+---------------+---------------------------------------------+---------------------------------------------+--------+
|     start_date|       end_date|                           start_station_name|                             end_station_name|duration|
+---------------+---------------+---------------------------------------------+---------------------------------------------+--------+
| 1/1/2014 13:42| 1/1/2014 14:36|          Mechanics Plaza (Market at Battery)|                       Embarcadero at Sansome|    3289|
| 1/1/2014 18:51| 1/1/2014 19:13|                       Embarcadero at Sansome|                                Market at 4th|    1286|
| 1/1/2014 19:48| 1/1/2014 20:01|                                Market at 4th|                     South Van Ness at Market|     795|
|1/10/2014 20:13|1/10/2014 20:17|                               Market at 10th|                           Powell Street BART|     235|
| 1/10/2014

Задача 4: Найти количество велосипедов в системе

In [8]:
unique_bikes = trips_df.select("bike_id").distinct().count()

print(f"Количество велосипедов в системе: {unique_bikes:,}")


Количество велосипедов в системе: 700


Задача 5: Найти пользователей, потративших на поездки более 3 часов

In [9]:
# Группируем по zip_code и суммируем время
user_total_time = trips_df.groupBy("zip_code") \
    .agg(
        sum("duration").alias("total_seconds"),
        count("id").alias("num_trips"),
        sum(when(col("subscription_type") == "Subscriber", 1).otherwise(0)).alias("subscriber_trips"),
        sum(when(col("subscription_type") == "Customer", 1).otherwise(0)).alias("customer_trips")
    ) \
    .filter(col("total_seconds") > 10800)  # 3 часа = 10800 секунд

# Сортируем по убыванию времени
user_total_time = user_total_time.orderBy(col("total_seconds").desc())

# Количество пользователей
users_over_3h = user_total_time.count()

print(f"Пользователи, потратившие на поездки более 3 часов: {users_over_3h}")

print("Первые 35 пользователей:")
user_total_time.show(35, truncate=False)

Пользователи, потратившие на поездки более 3 часов: 3661
Первые 35 пользователей:
+--------+-------------+---------+----------------+--------------+
|zip_code|total_seconds|num_trips|subscriber_trips|customer_trips|
+--------+-------------+---------+----------------+--------------+
|94107   |49757162     |78704    |76351           |2353          |
|nil     |45725550     |10682    |0               |10682         |
|NULL    |27723273     |6619     |0               |6619          |
|94105   |25596128     |42672    |41328           |1344          |
|94133   |21637675     |31359    |30320           |1039          |
|94102   |19128021     |19757    |18552           |1205          |
|94103   |19127388     |26673    |25434           |1239          |
|95531   |17270400     |1        |0               |1             |
|94111   |14244997     |21409    |20580           |829           |
|95112   |12742370     |11564    |10563           |1001          |
|94109   |12057128     |13989    |12890        